# Image Footprint & Geolocation Overlay**Source**: `grdl/example/geolocation/geolocation_overlay.py`## Learning Objectives- Compute image footprint (bounding polygon in lat/lon)- Sample pixel grid and overlay coordinates on imagery- Validate geolocation accuracy via visual inspection- Export coordinates for external mapping tools (Google Maps, QGIS)## Theory: Footprint Computation**Image footprint**: The geographic polygon enclosing the image extent.**GRDL workflow**:1. Sample corner + edge pixels → lat/lon via geolocation2. Construct polygon from boundary points3. Overlay on slippy map or export as GeoJSON**Use cases**:- Catalog indexing (spatial search)- Overlap detection (multi-image mosaics)- Mission planning (coverage verification)**Geolocation validation**: Sample a grid of pixels across the image and overlay both pixel (row, col) and geographic (lat, lon) coordinates to verify geolocation accuracy.## Data Setup**Path Configuration**: Set path to a SAR image (SICD or BIOMASS).For this example, use any SICD .nitf file or BIOMASS L1 directory.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass  # Not running in IPython

# Validate GRDL version
import grdl
from packaging.version import Version

REQUIRED_VERSION = '0.6.1'
current_version = Version(grdl.__version__)
required_version = Version(REQUIRED_VERSION)

if current_version < required_version:
    raise RuntimeError(
        f"GRDL {REQUIRED_VERSION}+ required. Found {grdl.__version__}. "
        f"Update with: pip install --upgrade grdl"
    )

print(f"✓ GRDL {grdl.__version__} (>= {REQUIRED_VERSION})")

In [ ]:
# Configuration (update paths to your data)SICD_PATH = '/path/to/your/sicd_file.nitf'FORMAT = 'sicd'  # or 'biomass'# Processing optionsCHIP_SIZE = 2048  # Crop large images to center chip (None = full image)GRID_SIZE = 5  # Grid points per axis (5x5 = 25 points)MARGIN_PCT = 5.0  # Margin from image edges (%)print(f"Configuration:")print(f"  Format: {FORMAT}")print(f"  Chip size: {CHIP_SIZE if CHIP_SIZE else 'full image'}")print(f"  Grid: {GRID_SIZE}x{GRID_SIZE} = {GRID_SIZE**2} points")print(f"  Margin: {MARGIN_PCT}%")

In [ ]:
import numpy as npimport matplotlib.pyplot as pltfrom pathlib import Pathfrom grdl.IO import get_readerfrom grdl.geolocation import create_geolocationfrom grdl.geolocation.chip import ChipGeolocation

In [ ]:
# Load image and build geolocation# For large images, extract a center chip to reduce memory and processing timewith get_reader(FORMAT, SICD_PATH) as reader:    metadata = reader.metadata    rows, cols = reader.get_shape()        print(f"Full image shape: {rows} x {cols}")        if CHIP_SIZE and (rows > CHIP_SIZE or cols > CHIP_SIZE):        # Crop to center chip        r_start = max(0, rows // 2 - CHIP_SIZE // 2)        c_start = max(0, cols // 2 - CHIP_SIZE // 2)        r_end = min(rows, r_start + CHIP_SIZE)        c_end = min(cols, c_start + CHIP_SIZE)                chip_data = reader.read_chip(r_start, r_end, c_start, c_end)        chip_offset = (r_start, c_start)                print(f"Reading center chip: {chip_data.shape[0]} x {chip_data.shape[1]}")        print(f"  Chip offset: row={r_start}, col={c_start}")    else:        chip_data = reader.read_full()        chip_offset = (0, 0)        print(f"Reading full image: {chip_data.shape}")# Build geolocation (auto-detect from reader)geo_full = create_geolocation(reader)# If chip was extracted, wrap geolocation with chip offsetif chip_offset != (0, 0):    geo = ChipGeolocation(geo_full, row_offset=chip_offset[0], col_offset=chip_offset[1])else:    geo = geo_fullprint(f"\nGeolocation type: {type(geo).__name__}")

In [ ]:
# Compute grid of sample points across the chipimg_rows, img_cols = chip_data.shape[:2]mr = int(img_rows * MARGIN_PCT / 100.0)mc = int(img_cols * MARGIN_PCT / 100.0)sample_rows = np.linspace(mr, img_rows - 1 - mr, GRID_SIZE, dtype=int)sample_cols = np.linspace(mc, img_cols - 1 - mc, GRID_SIZE, dtype=int)points = []for r in sample_rows:    for c in sample_cols:        # Convert chip coordinates to full-image coordinates        if isinstance(geo, ChipGeolocation):            full_r = float(r + chip_offset[0])            full_c = float(c + chip_offset[1])        else:            full_r, full_c = float(r), float(c)                try:            lat, lon, h = geo_full.image_to_latlon(full_r, full_c)            if np.isfinite(lat) and np.isfinite(lon):                points.append({                    'img_row': r, 'img_col': c,                    'full_row': full_r, 'full_col': full_c,                    'lat': lat, 'lon': lon, 'height': h,                })        except (ValueError, NotImplementedError):            continueprint(f"\nComputed {len(points)} grid points")print(f"\nSample points (first 5):")print(f"{'#':>3}  {'Row':>7}  {'Col':>7}  {'Latitude':>12}  {'Longitude':>12}  {'Height (m)':>10}")print("-" * 62)for i, p in enumerate(points[:5]):    print(f"{i+1:3d}  {p['full_row']:7.0f}  {p['full_col']:7.0f}  "          f"{p['lat']:12.6f}  {p['lon']:12.6f}  {p['height']:10.1f}")if len(points) > 5:    print(f"... ({len(points) - 5} more points)")

In [ ]:
# Convert to magnitude (dB) for visualizationdef to_db(arr):    return 20.0 * np.log10(np.abs(arr) + 1e-10)db = to_db(chip_data)vmin = np.nanpercentile(db, 2)vmax = np.nanpercentile(db, 98)print(f"\nImage statistics (dB):")print(f"  Min: {db.min():.1f} dB")print(f"  Max: {db.max():.1f} dB")print(f"  Mean: {db.mean():.1f} dB")print(f"  Display range: [{vmin:.1f}, {vmax:.1f}] dB (2nd-98th percentile)")

In [ ]:
# Visualize: overlay grid points with pixel and geographic coordinatesfig, ax = plt.subplots(figsize=(14, 12))ax.imshow(db, cmap='gray', vmin=vmin, vmax=vmax, aspect='auto')# Plot markerspx = [p['img_col'] for p in points]py = [p['img_row'] for p in points]ax.scatter(    px, py,    marker='o', c='cyan', s=80, edgecolors='yellow', linewidths=1.5,    label=f'{len(points)} sample points')# Annotate points with coordinates (every other point to reduce clutter)for i, p in enumerate(points[::2]):  # Skip every other point    label = f"({p['full_row']:.0f}, {p['full_col']:.0f})\n{p['lat']:.4f}°, {p['lon']:.4f}°"    ax.annotate(        label,        xy=(p['img_col'], p['img_row']),        xytext=(5, 5),        textcoords='offset points',        fontsize=8,        color='yellow',        bbox=dict(boxstyle='round,pad=0.3', facecolor='black', alpha=0.7)    )ax.set_title('Geolocation Overlay: Pixel (row, col) → Geographic (lat, lon)', fontsize=14)ax.legend(loc='upper right')plt.tight_layout()plt.show()

In [ ]:
# Compute image footprint (bounding polygon)# Sample corner + edge pixelsfootprint_pixels = []# Top edgefootprint_pixels.extend([(0, c) for c in range(0, img_cols, img_cols // 20)])# Right edgefootprint_pixels.extend([(r, img_cols - 1) for r in range(0, img_rows, img_rows // 20)])# Bottom edgefootprint_pixels.extend([(img_rows - 1, c) for c in range(img_cols - 1, -1, -(img_cols // 20))])# Left edgefootprint_pixels.extend([(r, 0) for r in range(img_rows - 1, -1, -(img_rows // 20))])footprint_coords = []for r, c in footprint_pixels:    try:        if isinstance(geo, ChipGeolocation):            full_r = float(r + chip_offset[0])            full_c = float(c + chip_offset[1])        else:            full_r, full_c = float(r), float(c)        lat, lon, _ = geo_full.image_to_latlon(full_r, full_c)        if np.isfinite(lat) and np.isfinite(lon):            footprint_coords.append((lon, lat))    except (ValueError, NotImplementedError):        continueprint(f"\nImage footprint: {len(footprint_coords)} boundary points")print(f"Bounding box:")lons = [c[0] for c in footprint_coords]lats = [c[1] for c in footprint_coords]print(f"  Latitude: [{min(lats):.6f}, {max(lats):.6f}]")print(f"  Longitude: [{min(lons):.6f}, {max(lons):.6f}]")

## Physical Interpretation**Grid overlay**: Each marker shows both pixel coordinates (row, col) and geographic coordinates (lat, lon). This allows visual verification that:- Geolocation is continuous (no discontinuities)- Coordinates increase monotonically (no flips)- Spacing is approximately uniform (no warping)**Footprint**: The bounding polygon defines the image's geographic extent. Use this for:- Catalog spatial indexing- Overlap detection with other images- Export to QGIS, Google Earth, or web maps**Export to GeoJSON** (optional):```pythonimport jsonfootprint_geojson = {    "type": "Feature",    "geometry": {        "type": "Polygon",        "coordinates": [footprint_coords]    },    "properties": {"format": FORMAT, "rows": rows, "cols": cols}}with open('footprint.geojson', 'w') as f:    json.dump(footprint_geojson, f, indent=2)```**Google Maps link** (center point):```pythoncenter_lat = (min(lats) + max(lats)) / 2center_lon = (min(lons) + max(lons)) / 2print(f"https://www.google.com/maps?q={center_lat},{center_lon}&t=k")```## Summary**GRDL patterns demonstrated**:- ✅ `create_geolocation(reader)` — auto-detect geolocation type- ✅ `ChipGeolocation` — chip offset wrapper- ✅ `image_to_latlon()` — pixel → lat/lon conversion- ✅ Footprint computation — boundary sampling**Validation workflow**: Sample grid → overlay coordinates → visual inspection → export footprint